# Team 10 · Member 8 · v3.1: v3 + collision energy and neutral losses (CASMI 2026)

**v3.1 = v3 with two richer MLP inputs:** the collision energy of each spectrum (mean, ramp width, present) and a neutral-loss block (precursor minus fragment mass, 500 bins of 1 Da), so the network sees losses such as water (18 Da) or a hexose (162 Da) directly. Everything else is v3.

**Why v3.** v2 (late fusion) scored 0.687 on the team split but **0.200** on the leaderboard. On the split the answer
is always a training structure in our pool; on the hidden test most answers are natural products that are not.
The score is then capped by **recall** (is the answer a candidate at all?) and by telling the real structure apart
from thousands of same-mass isomers. v3 adds, on top of v2's tested pipeline:

| Part | What it does |
|---|---|
| PubChem channel | for each molecule, every structure of the 105.9M PubChem tier within 10 ppm (capped by documentation), Morgan bits computed on the fly and scored with **our** fingerprint MLP; the best by log-likelihood, soft Tanimoto and fingerprint + documentation become extra candidates |
| `pop` prior | ln(1 + PubChem substances) + ln(1 + PubMed articles): well-documented structures first |
| `np` prior | COCONUT / LOTUS / NPAtlas membership (the hidden test comes from biological extracts) |
| hidden-like validation | each validation molecule is also scored as if its structure were unknown (removed from the training structures, kept only if COCONUT / PubChemLite / PubChem hold it; no library spectra) |
| tuning target | fusion weights tuned, 2-fold cross-fitted, on 0.2 × class 2 + 0.8 × unknown-structure copies (half of that on the natural-product proxy), instead of balanced MRR |

Ideas from the public CASMI 2026 notebooks (documentation prior and PubChem channel: Dmitrii Gluzdov and the
authors credited in *CASMI 26: From Spectra to Structures*; uniqueness rule: goodpjw2008 / omerfarukmerey).
The code here is our own; the datasets are public and credited below.

**Before you run: Add Input**
1. The competition data and the code dataset `casmi-m8-code` (**new version with `recall.py`**).
2. Member 7's `coconut-2026-11-lite` and `pubchemlite-2026-09`.
3. `dmitriigluzdov/casmi26-pubchem-popularity-prior` (PubChem tier and documentation arrays) and
   `dmitriigluzdov/casmi26-natural-product-knowledge-table` (LOTUS / NPAtlas).
   Not needed: `pubchem-2026-11` and `casmi26-attributed-v4g-reproduction-assets`.

**Settings:** GPU T4, **Internet off**. Expected runtime 1.5-3 h (the PubChem channel has time budgets; the test set
is shortlisted first). If the tier is not attached, v3 still runs with the priors on the v2 pool.

**Credits and licences.** PubChem tier and popularity prior: Dmitrii Gluzdov (PubChem data courtesy of the U.S.
National Library of Medicine). Natural-product table: Dmitrii Gluzdov (LOTUS CC0, NPAtlas CC BY-NC 4.0). COCONUT
(CC0) and PubChemLite (CC BY 4.0, Schymanski et al.) via Member 7. Competition data CC BY-NC 4.0.

In [ ]:
# ---- Settings ----
import glob
import os
import sys
import time

QUICK = False                       # True: small smoke run
N_VAL = 300 if QUICK else 3000      # validation molecules, as Member 7
MAX_QUERY_SPECTRA = 16
PPM = 10.0
PER_STRUCTURE = 2 if QUICK else 8
MAX_TRAIN = 1_200_000
FP_EPOCHS = 2 if QUICK else 20
FP_MAX_MIN = 150
USE_EXTRA = True                    # COCONUT + PubChemLite
USE_TIER = True                     # PubChem tier channel
CE_FEATURES = True                  # v3.1: collision energy (mean, ramp width, present) as MLP inputs
N_LOSS_BINS = 500                   # v3.1: neutral-loss block (precursor - fragment), e.g. 500 bins of 1 Da
TIER_W_MAX = 20_000                 # per window, best documented kept when larger
TIER_K_FP, TIER_K_MIX, TIER_POP_W = 60, 60, 0.25
TIER_TEST_MIN = 75                  # time budgets (minutes); test first
TIER_VAL_MIN = 20 if QUICK else 120
SKIP_PATHS = ("casmi26-pubchem-popularity-prior", "casmi26-attributed", "natural-product", "pubchem-2026")
OUT = "/kaggle/working"
RES = f"{OUT}/m8_results"
os.makedirs(RES, exist_ok=True)
T0 = time.time()


def stamp(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

In [ ]:
# ---- Code dataset: Member 3's casmi_pipeline.py + Member 8's ensemble_common ----
cp_hits = glob.glob("/kaggle/input/**/casmi_pipeline.py", recursive=True)  # old or new location in the repo
ens_hits = glob.glob("/kaggle/input/**/models/ensembles/ensemble_common/standalone/recall.py", recursive=True)
if not cp_hits or not ens_hits:
    raise FileNotFoundError("Attach the new version of the casmi-m8-code dataset (needs casmi_pipeline.py and "
                            "ensemble_common/standalone/recall.py).")
sys.path.insert(0, os.path.dirname(cp_hits[0]))
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(ens_hits[0]))))
print("casmi_pipeline:", cp_hits[0])
print("ensemble_common:", os.path.dirname(os.path.dirname(ens_hits[0])))

In [ ]:
# ---- Imports, RDKit, GPU ----
import importlib
import json
import subprocess

try:
    import rdkit  # noqa: F401
except ImportError:
    wheels = glob.glob("/kaggle/input/**/rdkit*.whl", recursive=True)
    if wheels:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", wheels[0]], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)
    importlib.invalidate_caches()

import casmi_pipeline as cp
import numpy as np
import pandas as pd
import rdkit
import torch
from ensemble_common.standalone import recall as R
from ensemble_common.standalone import teamsplit as T
from ensemble_common.standalone.candidates import find_candidate_files, load_external, pool_fingerprints
from ensemble_common.standalone.fpnet import PackedFingerprintMLP

CFG = T.FusionConfig(ppm=PPM, n_random=100 if QUICK else 300, n_loss_bins=N_LOSS_BINS)
print("v3.1 switches: collision energy", CE_FEATURES, "| neutral-loss bins", N_LOSS_BINS)
DATA = cp.find_data_dir()
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("rdkit", rdkit.__version__, "| torch", torch.__version__, "| GPU:", GPU, "| CPUs:", os.cpu_count())
print("data:", DATA)

In [ ]:
# ---- Split (as Member 7), pool = training structures + COCONUT + PubChemLite, prior files, PubChem tier ----
meta = cp.load_meta(DATA)
queries, lib_mask = cp.make_split(meta, max_query_spectra=MAX_QUERY_SPECTRA)
queries = cp.sample_molecules(queries, N_VAL)
truth = cp.truth_table(queries).reset_index(drop=True)
assert not lib_mask[queries.row_id.to_numpy()].any(), "validation spectra leaked into the library"
stamp(f"split: {len(truth):,} molecules (class 1: {(truth.cls == 1).sum()}, class 2: {(truth.cls == 2).sum()}, "
      f"np proxy: {(truth.query_lib == 'enveda-np-examples').sum()}), {len(queries):,} spectra")

import ensemble_common.standalone.candidates as candidates

candidates.SOURCES = ("coconut", "pubchemlite")
files = find_candidate_files("/kaggle/input") if USE_EXTRA else {}
files = {k: [f for f in v if not any(s in f for s in SKIP_PATHS)] for k, v in files.items()}
files = {k: v for k, v in files.items() if v}
print("extra candidate files:", {k: [os.path.basename(f) for f in v] for k, v in files.items()} or "none")
extra, extra_fp = load_external(files, n_bits=cp.FP_BITS, log=stamp) if files else (None, None)
ext_keys = frozenset(extra.key) if extra is not None else frozenset()
coconut_keys = frozenset(extra.key[extra.source == "coconut"]) if extra is not None else frozenset()
pool = cp.build_pool(meta, lib_mask, extra_candidates=extra)
pool_fp = pool_fingerprints(pool, extra, extra_fp, cp.FP_BITS)
key_index = pd.Series(np.arange(len(pool)), index=pool.key.to_numpy())
n_main = len(pool)
n_train_struct = int((pool.formula.astype(str) != "").sum())
stamp(f"pool: {n_main:,} structures ({n_train_struct:,} training, {n_main - n_train_struct:,} external)")


def first(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True), key=len)
    return hits[0] if hits else None


POP_CSV, NP_TABLE = first("pool_popularity.csv"), first("natural_products.parquet")
PCL_FILES = [f for f in files.get("pubchemlite", []) if f.lower().endswith(".csv")]
tier = R.PubChemTier.find("/kaggle/input") if USE_TIER else None
print("popularity:", POP_CSV, "| natural products:", NP_TABLE,
      "| PubChemLite:", [os.path.basename(f) for f in PCL_FILES])
print("PubChem tier:", f"{len(tier):,} structures, priors {sorted(tier.arrays)}" if tier else "not attached")

In [ ]:
# ---- One pass over train.parquet: centroid libraries + binned spectra (as v2) ----
test, test_path = cp.load_test(DATA)
test_mols = test.molecule_id.drop_duplicates().tolist()
val_mols = truth.molecule_id.tolist()
val_mass = queries.groupby("molecule_id").neutral_mass.median().reindex(val_mols)
test_mass = test.groupby("molecule_id").neutral_mass.median().reindex(test_mols)

train_rows = cp.select_train_rows(meta, lib_mask, per_structure=PER_STRUCTURE)
train_rows = train_rows[train_rows.inchikey14.isin(key_index.index)]
if len(train_rows) > MAX_TRAIN:
    train_rows = train_rows.sample(MAX_TRAIN, random_state=2026).sort_values("row_id")
train_rows = train_rows.reset_index(drop=True)

libs = T.build_libraries(cp, meta, lib_mask, pool, val_mass, test_mass, CFG)
mats = T.feature_pass(os.path.join(DATA, "train.parquet"), len(meta), libs,
                      {"train": train_rows.row_id.to_numpy(), "val": queries.row_id.to_numpy()}, CFG, log=stamp)
X_te = T.bin_file(test_path, len(test), CFG)
if CE_FEATURES:
    ce_tr = T.collision_energy(os.path.join(DATA, "train.parquet"), len(meta))
    ce_te = T.collision_energy(test_path, len(test))
    S_tr = T.side_features(cp, train_rows, ce_tr[train_rows.row_id.to_numpy()])
    S_va = T.side_features(cp, queries, ce_tr[queries.row_id.to_numpy()])
    S_te = T.side_features(cp, test, ce_te[test.row_id.to_numpy()])
    print(f"collision energy present: {ce_tr[:, 2].mean():.1%} of training rows, {ce_te[:, 2].mean():.1%} of test rows")
else:
    S_tr, S_va, S_te = (T.side_features(cp, f) for f in (train_rows, queries, test))
stamp(f"features: train {mats['train'].shape}, val {mats['val'].shape}, test {X_te.shape} "
      f"({len(test_mols)} test molecules)")

In [ ]:
# ---- Fingerprint MLP (as v2) ----
val_t = queries.true_key.map(key_index)
ok = val_t.notna().to_numpy()
model = PackedFingerprintMLP(epochs=FP_EPOCHS, max_minutes=FP_MAX_MIN)
model.fit(mats["train"], S_tr, train_rows.inchikey14.map(key_index).to_numpy(), pool_fp,
          val=(mats["val"][ok], S_va[ok], val_t[ok].astype(int).to_numpy()), log=stamp)
pd.DataFrame(model.history).to_csv(f"{RES}/fp_history.csv", index=False)
torch.save({k: v.cpu() for k, v in model.net.state_dict().items()}, f"{RES}/fp_mlp.pt")
P_va = model.predict(mats["val"], S_va)
P_te = model.predict(X_te, S_te)
del mats["train"]
import gc

gc.collect()
torch.cuda.empty_cache()
stamp("MLP trained")

In [ ]:
# ---- PubChem channel: shortlists per molecule (test first), extended pool, pop / np priors ----
tier_info = {}
try:
    opt = dict(ppm=PPM, w_max=TIER_W_MAX, k_fp=TIER_K_FP, k_mix=TIER_K_MIX, pop_w=TIER_POP_W, log=stamp)
    hits_te, fr_te, fp_te = R.tier_shortlists(tier, test_mols, test_mass.to_numpy(),
                                              R.molecule_probs(test, P_te, test_mols),
                                              max_minutes=TIER_TEST_MIN, **opt)
    hits_va, fr_va, fp_va = R.tier_shortlists(tier, val_mols, val_mass.to_numpy(),
                                              R.molecule_probs(queries, P_va, val_mols),
                                              max_minutes=TIER_VAL_MIN, **opt)
    frame = pd.concat([fr_te, fr_va], ignore_index=True)
    frame_fp = np.concatenate([fp_te, fp_va])
    keep = ~frame.tier_row.duplicated().to_numpy()
    frame, frame_fp = frame[keep].reset_index(drop=True), frame_fp[keep]
    pool_all, fp_all, mol_idx = R.extend_pool(pool, pool_fp, tier, {**hits_te, **hits_va}, frame, frame_fp)
    stamp(f"pool extended: {len(pool_all):,} structures")
    tier_info = {"test_molecules": len(hits_te), "val_molecules": len(hits_va), "unique_structures": int(len(frame)),
                 "added_to_pool": int(len(pool_all) - n_main)}
except Exception as err:  # noqa: BLE001 - v3 still runs without the tier
    print("PubChem channel failed, continuing without it:", repr(err))
    pool_all, fp_all, mol_idx = R.extend_pool(pool, pool_fp, None, {}, pd.DataFrame(), None)
    tier_info = {"error": repr(err)}
stamp("computing priors")
pop, npf = R.candidate_priors(pool_all, coconut_keys, POP_CSV, NP_TABLE, PCL_FILES, tier,
                              pool_all.tier_row.to_numpy(), log=stamp)
tk = dict(zip(truth.molecule_id, truth.true_key, strict=True))
keys_all = pool_all.key.to_numpy()
in_short = np.mean([tk[m] in set(keys_all[mol_idx.get(m, [])]) for m in val_mols])
tier_info["val_truth_in_shortlist"] = round(float(in_short), 4)
tier_info["val_truth_in_external_sources"] = round(float(np.mean([tk[m] in ext_keys for m in val_mols])), 4)
stamp(f"pool with shortlists: {len(pool_all):,} structures | {json.dumps(tier_info)}")

In [ ]:
# ---- Validation: standard copies + unknown-structure copies, fusion tuned on the hidden-like objective ----
args = (cp, queries, mats["val"], P_va, libs.val, pool_all, fp_all, pop, npf, n_main, CFG)
sc_std = R.score_v3(*args, mols=val_mols, extra_idx=mol_idx, truth_keys=tk)
sc_nov = R.score_v3(*args, mols=val_mols, extra_idx=mol_idx, truth_keys=tk, novel=True, ext_keys=ext_keys)
stamp(f"validation scored: truth among candidates {np.mean(sc_std.truth >= 0):.3f} (standard), "
      f"{np.mean(sc_nov.truth >= 0):.3f} (unknown structure)")
rep = R.tune_v3(cp, sc_std, sc_nov, truth, pool_all, n_main, CFG, ext_keys=ext_keys, log=stamp)
best = rep["best"]

# uniqueness rule (every molecule is a different structure): keep it only if it helps on validation
f_std = rep["val_scores"].get(best, sc_std.src.get(best))
r_std = R.rankings_v3(cp, sc_std, f_std, pool_all, n_main)
off = sc_std.offsets
lead = {m: float(f_std[off[i]:off[i + 1]].max()) for i, m in enumerate(sc_std.mols) if off[i + 1] > off[i]}
t_plain = cp.evaluate(r_std, truth, keys_all)
t_uniq = cp.evaluate(R.unique_leaders(r_std, lead), truth, keys_all)
USE_UNIQ = bool(t_uniq.loc["class 2", "MRR@25"] + t_uniq.loc["class 1", "MRR@25"]
                > t_plain.loc["class 2", "MRR@25"] + t_plain.loc["class 1", "MRR@25"])
b1 = {m: cp.rank_b1(pool, val_mass[m], PPM) for m in val_mols}
t_b1 = cp.evaluate(b1, truth, keys_all)
pd.set_option("display.width", 250)
print("Validation (fusion rows cross-fitted). objective = 0.2 x class 2 + 0.8 x unknown-structure copies:")
print(rep["summary"].to_string(index=False))
print("B1 (mass only):", t_b1["MRR@25"].to_dict())
print(f"uniqueness rule: class1+class2 {t_plain.loc[['class 1', 'class 2'], 'MRR@25'].sum():.4f} -> "
      f"{t_uniq.loc[['class 1', 'class 2'], 'MRR@25'].sum():.4f} | used for test: {USE_UNIQ}")
print("chosen:", best, json.dumps(R.describe(rep["params"][best]), default=str))
r_val = R.unique_leaders(r_std, lead) if USE_UNIQ else r_std  # cross-fitted validation ranking, as submitted

In [ ]:
# ---- Test: v3 scores, chosen fusion, uniqueness rule, submission.csv (B1 fallback per molecule) ----
test_cov = {}
try:
    sc_te = R.score_v3(cp, test, X_te, P_te, libs.test, pool_all, fp_all, pop, npf, n_main, CFG,
                       mols=test_mols, extra_idx=mol_idx)
    f_te = R.apply_v3(rep, sc_te, CFG)
    test_rank = R.rankings_v3(cp, sc_te, f_te, pool_all, n_main)
    if USE_UNIQ:
        off = sc_te.offsets
        lead = {m: float(f_te[off[i]:off[i + 1]].max()) for i, m in enumerate(sc_te.mols) if off[i + 1] > off[i]}
        test_rank = R.unique_leaders(test_rank, lead)
    test_cov = T.coverage(sc_te)
    test_cov["from_pubchem_channel"] = round(float(np.mean([any(j >= n_main for j in r[:25])
                                                            for r in test_rank.values()])), 4)
    pd.DataFrame({"query_id": np.asarray(sc_te.mols, dtype=object)[sc_te.seg], "inchikey14": keys_all[sc_te.cand],
                  "score": f_te, **{s: sc_te.src[s] for s in R.SOURCES}}).to_parquet(f"{RES}/test_scores.parquet",
                                                                                   index=False)
except Exception as err:  # noqa: BLE001 - the hidden-test rerun must always write a submission
    print("test scoring failed, using B1 for every molecule:", repr(err))
    test_rank, test_cov = {}, {"error": repr(err)}
fallback = [m for m in test_mols if not test_rank.get(m)]
for m in fallback:
    test_rank[m] = cp.rank_b1(pool, test_mass[m], PPM)
sub = cp.write_submission(test_rank, pool_all, test, path=f"{OUT}/submission.csv")
try:  # the exact submitted ranking (filler and B1 fallback included); *_scores.parquet holds raw window scores
    T.ranking_table(test_rank, pool_all).to_parquet(f"{RES}/test_ranking.parquet", index=False)
except Exception as err:  # noqa: BLE001 - an export must never block the submission
    print("ranking export skipped:", repr(err))
stamp(f"submission.csv: {len(sub)} rows, {len(fallback)} on the B1 fallback | test coverage {test_cov}")

In [ ]:
# ---- Save results for the report and slides ----
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

rep["summary"].to_csv(f"{RES}/validation_report.csv", index=False)
try:  # validation ranking in the team format (rank_candidates reproduces it exactly)
    T.ranking_table(r_val, pool_all).to_parquet(f"{RES}/val_ranking.parquet", index=False)
except Exception as err:  # noqa: BLE001
    print("ranking export skipped:", repr(err))
for name, table in rep["tables"].items():
    table.to_csv(f"{RES}/val_{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
summary = {
    "date": time.strftime("%Y-%m-%d %H:%M"), "runtime_min": round((time.time() - T0) / 60, 1), "gpu": GPU,
    "quick": QUICK, "chosen": best, "chosen_params": R.describe(rep["params"][best]),
    "all_params": {k: R.describe(v) for k, v in rep["params"].items()},
    "objective": R.OBJECTIVE, "uniqueness_rule": USE_UNIQ, "tier": tier_info,
    "split": {"molecules": int(len(truth)), "class1": int((truth.cls == 1).sum()),
              "class2": int((truth.cls == 2).sum()), "np_proxy": int((truth.query_lib == "enveda-np-examples").sum())},
    "pool": {"main": n_main, "with_shortlists": int(len(pool_all)), "training_structures": n_train_struct},
    "coverage": {"val_standard": round(float(np.mean(sc_std.truth >= 0)), 4),
                 "val_unknown_structure": round(float(np.mean(sc_nov.truth >= 0)), 4),
                 "test": test_cov, "test_b1_fallback": len(fallback)},
    "validation": rep["summary"].set_index("method").drop(columns=["kind"]).to_dict("index"),
    "b1": t_b1["MRR@25"].to_dict(), "fp_history": model.history[-3:],
    "config": {"ppm": PPM, "n_val": N_VAL, "tier_w_max": TIER_W_MAX, "tier_k_fp": TIER_K_FP, "tier_k_mix": TIER_K_MIX,
               "tier_pop_w": TIER_POP_W, "ce_features": CE_FEATURES, "n_loss_bins": N_LOSS_BINS,
               "fusion": vars(CFG)},
}
with open(f"{RES}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

s = rep["summary"].set_index("method")[["class 2", "novel all", "novel np", "objective"]].astype(float)
ax = s.iloc[::-1].plot.barh(figsize=(8, 0.45 * len(s) + 2), color=["#eb6834", "#7a5cc7", "#1baf7a", "#555555"])
ax.set_xlabel("validation MRR@25 (fusion rows cross-fitted)")
ax.set_title("Member 8 v3: sources and fusion on the hidden-like objective", loc="left")
plt.tight_layout()
plt.savefig(f"{RES}/validation_report.png", dpi=150)
plt.show()
for root, _, names in os.walk(OUT):
    for n in sorted(names):
        p = os.path.join(root, n)
        print(f"{os.path.getsize(p) / 1e6:8.2f} MB  {p}")
stamp("done")